# Load-out-of-store vs direct-path timing (+ chunking behaviour)

How much does going *through the store* cost versus just reading a file that is already
sitting at a known path? This times **every step** of retrieving an artifact through the
API —

1. **finding** the node (by id, and by metadata search),
2. **discovering** its artifact with `.artifacts()` (which, for a packed store, also
   *reassembles* the file from the chunk pool),
3. **reading / parsing** the returned path,

— and compares the total against `pd.read_csv(path)` on a plain copy of the same file.
Because reads route through a session cache, the store path has two regimes: **cold**
(first touch — decompress from the pool) and **warm** (cache hit — a plain file read).
Both are measured, along with the write-side penalty by file type and size.

Part 2 reruns the 0.1.x chunking-behaviour checks (sub-file sharing, incompressible
data, the read-cache lifecycle and the edge cases) on the SQLite backend.

Re-run top to bottom; the stores are wiped at the start.

In [1]:
import shutil
import tempfile
import time
from pathlib import Path
import numpy as np
import pandas as pd
import ancestree

WORK = Path(tempfile.mkdtemp(prefix="ancestree-timing-"))
ROOT = WORK / "load_benchmark_store"  # the lineage store
BASE = WORK / "load_benchmark_direct"  # plain copies of the same files, no store
BASE.mkdir(parents=True)

store = ancestree.LineageStore(ROOT, dedup=True, chunk=True)


def clear_read_cache(s):
    # 0.1.x exposed clear_cache() publicly; the cache is session-scoped and
    # automatic now, so faking a "new session" for COLD timings reaches into
    # the internals deliberately.
    s._chunks.clear_cache()


def timed(label, fn, reps=50):
    ts = []
    for _ in range(reps):
        t = time.perf_counter()
        fn()
        ts.append((time.perf_counter() - t) * 1000)
    print(f"{label:<44} median {np.median(ts):8.3f} ms   (best {min(ts):.3f})")
    return float(np.median(ts))

## 1. Build a store with real artifacts (+ plain copies for the baseline)

Most nodes are metadata-only (to give the search a realistic population); a handful hold
a ~2 MB CSV. Each artifact is also written verbatim to `BASE/` so we can read the
*identical bytes* directly, with no store involved.

In [2]:
N_TOTAL = 1000  # nodes in the store (search covers the whole population)
N_ART = 30  # of those, how many hold a real artifact we will load
ROWS = 95_000  # ~2 MB per CSV
rng = np.random.default_rng(0)


def make_csv(nrows):
    return pd.DataFrame(
        {
            "ts": np.arange(nrows),
            "a": rng.normal(size=nrows).round(4),
            "b": rng.normal(size=nrows).round(4),
            "label": rng.integers(0, 3, nrows),
        }
    )


target_ids, direct_paths = [], []
t0 = time.perf_counter()
for i in range(N_TOTAL):
    with store.create_node(step_type="run") as n:
        n.add_meta("run_id", i)
        n.add_meta("accuracy", round(float(rng.random()), 4))
        if i < N_ART:
            df = make_csv(ROWS)
            df.to_csv(n / "data.csv", index=False)  # stored (chunked)
            p = BASE / f"run_{i}.csv"
            df.to_csv(p, index=False)  # plain copy
            target_ids.append(n.node_id)
            direct_paths.append(p)

art_mb = direct_paths[0].stat().st_size / 1e6
print(
    f"built {N_TOTAL} nodes ({N_ART} with ~{art_mb:.1f} MB CSVs) in {time.perf_counter() - t0:.0f}s"
)
# Nodes are rows, not folders: at rest the store is just the database.
print("store root holds only:", sorted(p.name for p in ROOT.iterdir()))

built 1000 nodes (30 with ~2.2 MB CSVs) in 52s
store root holds only: ['.scratch', 'ancestree.db', 'ancestree.db-shm', 'ancestree.db-wal']


## 2. Baseline — the file is already at a known path

No store, no lineage, no decompression: just read the bytes / parse the CSV sitting on
disk. This is the bar everything else is compared against.

In [3]:
direct_bytes = timed("direct  path.read_bytes()", lambda: direct_paths[0].read_bytes())
direct_parse = timed("direct  pd.read_csv(path)", lambda: pd.read_csv(direct_paths[0]))

direct  path.read_bytes()                    median    0.086 ms   (best 0.085)


direct  pd.read_csv(path)                    median   16.244 ms   (best 15.905)


## 3. Retrieve through the store, by known id — every step

You already know the node id. Time each step: `get` (was `get_node`), `.artifacts()`
(discovers the file and, on a cold cache, reassembles it from chunks), then
`pd.read_csv` on the returned path. Cold clears the cache first so every artifact is a
fresh miss; warm runs straight after, all cache hits.

In [4]:
def breakdown_by_id(clear):
    if clear:
        clear_read_cache(store)
    g = a = r = 0.0
    for tid in target_ids:
        t = time.perf_counter()
        node = store.get(tid)
        g += time.perf_counter() - t
        t = time.perf_counter()
        path = node.artifacts("data.csv")[0]
        a += time.perf_counter() - t
        t = time.perf_counter()
        _ = pd.read_csv(path)
        r += time.perf_counter() - t
    n = len(target_ids)
    return g / n * 1000, a / n * 1000, r / n * 1000


id_cold = breakdown_by_id(clear=True)  # .artifacts() decompresses from the pool
id_warm = breakdown_by_id(clear=False)  # .artifacts() is a cache hit
for label, (g, a, r) in [("COLD (cache miss)", id_cold), ("WARM (cache hit)", id_warm)]:
    print(
        f"{label:<20}  get {g:7.3f}  | artifacts() {a:7.3f}  | read_csv {r:7.3f}  | TOTAL {g + a + r:7.3f} ms"
    )

COLD (cache miss)     get   0.050  | artifacts()   7.364  | read_csv  16.603  | TOTAL  24.018 ms
WARM (cache hit)      get   0.033  | artifacts()   0.110  | read_csv  16.282  | TOTAL  16.425 ms


## 4. Retrieve through the store, by metadata search — every step

The realistic case where you *don't* know the id: `find(run_id=...)` (was `find_node`)
queries the database, then the same discover + read. Shows what the search itself adds
on top of section 3.

In [5]:
def breakdown_by_search(clear):
    if clear:
        clear_read_cache(store)
    s = a = r = 0.0
    for i, _ in enumerate(target_ids):
        t = time.perf_counter()
        node = store.find(run_id=i)[0]
        s += time.perf_counter() - t
        t = time.perf_counter()
        path = node.artifacts("data.csv")[0]
        a += time.perf_counter() - t
        t = time.perf_counter()
        _ = pd.read_csv(path)
        r += time.perf_counter() - t
    n = len(target_ids)
    return s / n * 1000, a / n * 1000, r / n * 1000


se_cold = breakdown_by_search(clear=True)
se_warm = breakdown_by_search(clear=False)
for label, (s, a, r) in [("COLD", se_cold), ("WARM", se_warm)]:
    print(
        f"{label:<6} find {s:7.3f}  | artifacts() {a:7.3f}  | read_csv {r:7.3f}  | TOTAL {s + a + r:7.3f} ms"
    )

COLD   find   0.069  | artifacts()   7.197  | read_csv  16.542  | TOTAL  23.808 ms
WARM   find   0.051  | artifacts()   0.107  | read_csv  16.217  | TOTAL  16.375 ms


## 5. The convenience path: `node / "data.csv"`

When you know the filename you skip `.artifacts()` and index it directly. Same
materialization underneath.

In [6]:
def by_slash(clear):
    if clear:
        clear_read_cache(store)
    tot = 0.0
    for tid in target_ids:
        t = time.perf_counter()
        _ = pd.read_csv(store.get(tid) / "data.csv")
        tot += time.perf_counter() - t
    return tot / len(target_ids) * 1000


print(
    f"node / 'data.csv' then read  COLD {by_slash(True):7.3f} ms   WARM {by_slash(False):7.3f} ms"
)

node / 'data.csv' then read  COLD  23.834 ms   WARM  16.324 ms


## 6. Summary — store overhead over a direct path read

In [7]:
def row(name, total):
    print(f"  {name:<34} {total:8.3f} ms   ({total / direct_parse:5.1f}x direct)")


print(f"Reading one ~{art_mb:.1f} MB CSV ({len(target_ids)} samples averaged)\n")
print(f"  {'direct path.read_bytes()':<34} {direct_bytes:8.3f} ms   (raw I/O floor)")
print(f"  {'direct pd.read_csv(path)':<34} {direct_parse:8.3f} ms   (baseline)")
row("store by id   — COLD (first touch)", sum(id_cold))
row("store by id   — WARM (cache hit)", sum(id_warm))
row("store by search — COLD", sum(se_cold))
row("store by search — WARM", sum(se_warm))
print()
print(
    f"  decompression cost (artifacts COLD - WARM): {id_cold[1] - id_warm[1]:.3f} ms / artifact"
)
print(
    f"  metadata search cost (find):                {se_cold[0]:.3f} ms over {N_TOTAL} nodes"
)
print(f"  record load          (get):                 {id_cold[0]:.3f} ms")
print()
print("Takeaways:")
print(
    " - WARM store reads ~= a direct read: once cached, the store adds almost nothing."
)
print(
    " - The COLD premium is the chunk decompression, paid once per artifact per session."
)
print(
    " - Finding a node (record load or SQL search) is sub-millisecond and dwarfed by I/O."
)

Reading one ~2.2 MB CSV (30 samples averaged)

  direct path.read_bytes()              0.086 ms   (raw I/O floor)
  direct pd.read_csv(path)             16.244 ms   (baseline)
  store by id   — COLD (first touch)   24.018 ms   (  1.5x direct)
  store by id   — WARM (cache hit)     16.425 ms   (  1.0x direct)
  store by search — COLD               23.808 ms   (  1.5x direct)
  store by search — WARM               16.375 ms   (  1.0x direct)

  decompression cost (artifacts COLD - WARM): 7.254 ms / artifact
  metadata search cost (find):                0.069 ms over 1000 nodes
  record load          (get):                 0.050 ms

Takeaways:
 - WARM store reads ~= a direct read: once cached, the store adds almost nothing.
 - The COLD premium is the chunk decompression, paid once per artifact per session.
 - Finding a node (record load or SQL search) is sub-millisecond and dwarfed by I/O.


## 7. Read/write penalty by data type and size

The headline table: for a spread of file types and sizes, how long does it take to
**write** and **read** the file *directly* (a plain path) versus *through the store*?

- **W store** = the whole `create_node` block: the loose-file write **plus** the
  synchronous ingest at block exit (chunk + hash + compress + commit). Packing used to
  be deferred to a background worker in 0.1.x; it is synchronous and atomic now, so
  this column is the full price you actually wait for.
- **R cold** = first touch in a session (reassemble every chunk → cache).
- **R warm** = a second read in the same session (cache hit).

All `x` columns are the multiple over the *direct* operation. One sizing footnote:
these are decimal MB, so the "64 MB" arrays (64×10⁶ bytes) land just *under* the
64 MiB large-file threshold. Of the rows below, only the ~157 MB csv actually
crosses it — skipping the pure-Python chunking loop for fixed boundaries at C
speed — which is why its write multiple *falls* while the binary formats keep
paying the full per-byte rate.

In [8]:
import pickle

DT_ROOT = WORK / "dt_bench_store"
DT_BASE = WORK / "dt_bench_direct"
DT_BASE.mkdir(parents=True)
dt_store = ancestree.LineageStore(DT_ROOT, dedup=False, chunk=True)
dt_rng = np.random.default_rng(0)


def _med(fn, reps):
    ts = []
    for _ in range(reps):
        t = time.perf_counter()
        fn()
        ts.append((time.perf_counter() - t) * 1000)
    return float(np.median(ts))


def _npy_w(p, d):
    with open(p, "wb") as f:
        np.save(f, d)


def _npy_r(p):
    with open(p, "rb") as f:
        return np.load(f)


def _npz_w(p, d):
    with open(p, "wb") as f:
        np.savez_compressed(f, a=d)


def _npz_r(p):
    with open(p, "rb") as f:
        return np.load(f)["a"]


FORMATS = {
    "bytes": dict(
        fn="d.bin",
        make=lambda a: a.tobytes(),
        w=lambda p, d: Path(p).write_bytes(d),
        r=lambda p: Path(p).read_bytes(),
    ),
    "npy": dict(fn="d.npy", make=lambda a: a, w=_npy_w, r=_npy_r),
    "npz": dict(fn="d.npz", make=lambda a: a, w=_npz_w, r=_npz_r),
    "csv": dict(
        fn="d.csv",
        make=lambda a: pd.DataFrame(
            a[: len(a) // 4 * 4].reshape(-1, 4), columns=list("abcd")
        ),
        w=lambda p, d: d.to_csv(p, index=False),
        r=lambda p: pd.read_csv(p),
    ),
    "pickle": dict(
        fn="d.pkl",
        make=lambda a: a,
        w=lambda p, d: Path(p).write_bytes(pickle.dumps(d)),
        r=lambda p: pickle.loads(Path(p).read_bytes()),
    ),
}
SIZES_MB = [
    0.25,
    1,
    4,
    16,
    64,
]  # underlying float64 array size; file sizes differ per format

rows = []
for size_mb in SIZES_MB:
    arr = dt_rng.standard_normal(int(size_mb * 1e6 / 8))
    for name, F in FORMATS.items():
        data = F["make"](arr)
        dpath = DT_BASE / f"{name}_{size_mb}_{F['fn']}"
        wd = _med(lambda: F["w"](dpath, data), 5)  # direct write
        file_mb = dpath.stat().st_size / 1e6
        rd = _med(lambda: F["r"](dpath), 7)  # direct read
        t0 = time.perf_counter()  # store write = the whole block, ingest included
        with dt_store.create_node(step_type="blob") as node:
            F["w"](node / F["fn"], data)
        ws = (time.perf_counter() - t0) * 1000
        nid = node.node_id

        def _cold():
            clear_read_cache(dt_store)
            return F["r"](dt_store.get(nid) / F["fn"])

        rc = _med(_cold, 5)  # cold read (materialize + parse)
        rw = _med(
            lambda: F["r"](dt_store.get(nid) / F["fn"]), 7
        )  # warm read (cache hit)
        rows.append(
            dict(type=name, file_mb=file_mb, wd=wd, ws=ws, rd=rd, rc=rc, rw=rw)
        )
    print(f"  done size {size_mb} MB")
print("measurements complete")

  done size 0.25 MB


  done size 1 MB


  done size 4 MB


  done size 16 MB


  done size 64 MB
measurements complete


### The table

In [9]:
hdr = (
    f"{'type':7} {'file MB':>8} | {'W direct':>9} {'W store':>9} {'W x':>6} | "
    f"{'R direct':>9} {'R cold':>8} {'R warm':>8} {'cold x':>7} {'warm x':>7}"
)
print(hdr)
print("-" * len(hdr))
for r in rows:
    print(
        f"{r['type']:7} {r['file_mb']:8.2f} | {r['wd']:9.2f} {r['ws']:9.2f} "
        f"{r['ws'] / r['wd']:5.0f}x | {r['rd']:9.2f} {r['rc']:8.2f} {r['rw']:8.2f} "
        f"{r['rc'] / r['rd']:6.0f}x {r['rw'] / r['rd']:6.1f}x"
    )

print("\nReadings are milliseconds (median; W store is a single block per row).")

type     file MB |  W direct   W store    W x |  R direct   R cold   R warm  cold x  warm x
-------------------------------------------------------------------------------------------
bytes       0.25 |      0.14     68.32   503x |      0.04     1.62     0.07     45x    2.0x
npy         0.25 |      0.14     61.11   450x |      0.07     1.78     0.10     27x    1.5x
npz         0.24 |      7.75     70.23     9x |      0.77     1.71     0.80      2x    1.0x
csv         0.61 |     23.23    154.06     7x |      2.97     5.58     3.01      2x    1.0x
pickle      0.25 |      0.14     62.94   464x |      0.04     1.64     0.07     38x    1.6x
bytes       1.00 |      0.24    174.86   718x |      0.06     4.74     0.09     84x    1.5x
npy         1.00 |      0.29    140.85   485x |      0.08     4.89     0.12     58x    1.4x
npz         0.96 |     31.10    195.19     6x |      2.78     4.92     2.82      2x    1.0x
csv         2.45 |     92.33    514.78     6x |     11.04    20.04    11.15     

### Takeaways

In [10]:
fast = [r for r in rows if r["type"] in ("bytes", "npy", "pickle") and r["file_mb"] < 60]
slow = [r for r in rows if r["type"] == "csv"]
big = [r for r in rows if r["file_mb"] >= 60]
print("WRITE penalty is the big one below the large-file threshold:")
print(
    f"  ingest turns a ~{min(r['wd'] for r in fast):.2f}-{max(r['wd'] for r in fast):.1f} ms direct write into"
)
print(
    f"  {min(r['ws'] for r in fast):.0f}-{max(r['ws'] for r in fast):.0f} ms — up to {max(r['ws'] / r['wd'] for r in fast):.0f}x — the pure-Python"
)
print("  chunking loop runs at a few MB/s, so the cost scales with megabytes.")
if big:
    print()
    print("The biggest rows straddle the 64 MiB threshold (sizes above are decimal MB) - only csv crossed it, hence its multiple collapsing:")
    for r in big:
        print(f"  {r['type']:7} {r['file_mb']:6.1f} MB: W store {r['ws']:.0f} ms ({r['ws'] / r['wd']:.0f}x direct)")
print()
print("READ penalty depends entirely on the format:")
print(
    f"  fast binary (npy/bytes/pickle): cold read up to {max(r['rc'] / r['rd'] for r in fast):.0f}x direct"
)
print("     (the reassembly dwarfs the ~0-1 ms raw read), warm ~1x once cached.")
print(
    f"  slow text (csv): cold only ~{max(r['rc'] / r['rd'] for r in slow):.1f}x — the parse hides the reassembly."
)
print()
print("So: the store is cheap when your I/O is already slow (csv), and expensive")
print("relative to fast formats (npy) — especially on WRITE. Warm re-reads are ~free.")
print()
print("dt store stats:", {k: v for k, v in dt_store.stats().items() if k in ('nodes', 'chunks', 'dedup_ratio')})

WRITE penalty is the big one below the large-file threshold:
  ingest turns a ~0.14-4.7 ms direct write into
  61-2405 ms — up to 718x — the pure-Python
  chunking loop runs at a few MB/s, so the cost scales with megabytes.

The biggest rows straddle the 64 MiB threshold (sizes above are decimal MB) - only csv crossed it, hence its multiple collapsing:
  bytes     64.0 MB: W store 9391 ms (647x direct)
  npy       64.0 MB: W store 7190 ms (386x direct)
  npz       61.5 MB: W store 10508 ms (5x direct)
  csv      157.1 MB: W store 22072 ms (4x direct)
  pickle    64.0 MB: W store 7066 ms (286x direct)

READ penalty depends entirely on the format:
  fast binary (npy/bytes/pickle): cold read up to 113x direct
     (the reassembly dwarfs the ~0-1 ms raw read), warm ~1x once cached.
  slow text (csv): cold only ~1.9x — the parse hides the reassembly.

So: the store is cheap when your I/O is already slow (csv), and expensive
relative to fast formats (npy) — especially on WRITE. Warm re-reads

---

# Part 2 — chunking behaviour & the session read cache

The 0.1.x stress checks, re-run on the SQLite backend: sub-file dedup efficiency,
incompressible data, and the read-cache lifecycle (the store stays packed; reads route
through a session cache in the system temp directory).

## Sub-file sharing: many near-identical large CSVs

In [11]:
s = ancestree.LineageStore(WORK / "near", chunk=True)
rng = np.random.default_rng(0)
base = pd.DataFrame({"x": np.arange(80_000), "y": rng.normal(size=80_000).round(4)})
with s.create_node(step_type="ingest") as ing:
    base.to_csv(ing / "data.csv", index=False)
for i in range(8):  # 8 versions, each a tiny edit
    v = base.copy()
    v.loc[i * 1000, "y"] = 999.0
    with s.create_node(step_type="clean", parent=ing) as c:
        v.to_csv(c / "data.csv", index=False)

stats = s.stats()
one_csv = len(base.to_csv(index=False).encode())
print(
    f"9 ~identical CSVs | storing whole: {9 * one_csv / 1e6:.1f} MB | pool: {stats['chunk_stored_bytes'] / 1e6:.2f} MB"
)
print(f"dedup ratio (logical / stored): {stats['dedup_ratio']}")
s.close()

9 ~identical CSVs | storing whole: 9.5 MB | pool: 0.47 MB
dedup ratio (logical / stored): 20.424


## Incompressible data — exact chunk matching can't help, deltas can

0.1.x expected the pool to hold all ~1.5 MB here (three independent random blobs share
nothing). With Layer 2 the answer depends on similarity: three *independent* blobs
still share nothing — but had they been versions of each other, the delta layer would
have collapsed them (that case is exactly what section 7's random-binary rows and the
benchmark in `benchmarks/RESULTS.md` measure).

In [12]:
import os

s2 = ancestree.LineageStore(WORK / "random", chunk=True)
for i in range(3):
    with s2.create_node(step_type="blob") as n:
        (n / "r.bin").write_bytes(os.urandom(500_000))
print(f"3 x 500KB random: pool {s2.stats()['chunk_stored_bytes'] / 1e6:.2f} MB (~1.5 MB expected, no sharing)")
s2.close()

3 x 500KB random: pool 1.50 MB (~1.5 MB expected, no sharing)


## The read cache — the store stays packed; the path is node-scoped

0.1.x kept a per-node folder and checked the artifact never re-appeared "loose" inside
it. Nodes have no folders at all now — the equivalent guarantee is that the store root
holds nothing but the database, and reads hand back a path in the session cache (system
temp), regenerated from the pool on demand.

In [13]:
s3 = ancestree.LineageStore(WORK / "cache", chunk=True)
with s3.create_node(step_type="x") as n:
    (n / "results/clean.csv").write_text("a,b\n1,2\n")
print("store root after write:", sorted(p.name for p in s3.root.iterdir()))

p = s3.get(n.node_id) / "results/clean.csv"  # transparent read
print("returned path is in the session cache:", str(p).startswith(tempfile.gettempdir()))
print("read back:", repr(p.read_text()))

store root after write: ['.scratch', 'ancestree.db', 'ancestree.db-shm', 'ancestree.db-wal']
returned path is in the session cache: True
read back: 'a,b\n1,2\n'


In [14]:
# Cache hit within a session, then clear -> regenerates from the pool
p2 = s3.get(n.node_id) / "results/clean.csv"
print("same cached path reused:", p == p2)
clear_read_cache(s3)
print(
    "after clearing the cache, still readable:",
    (s3.get(n.node_id) / "results/clean.csv").read_text() == "a,b\n1,2\n",
)
s3.close()

same cached path reused: True
after clearing the cache, still readable: True


**Cross-session note:** the cache is wiped when the kernel exits, so the *first* read in
a new session re-decompresses from the pool (the store itself never re-inflates).
Re-reads within a session are free.

## Edge cases that work correctly

In [15]:
s4 = ancestree.LineageStore(WORK / "edges", chunk=True)
with s4.create_node(step_type="x") as n:
    (n / "empty.bin").write_bytes(b"")  # 0-byte file
    (n / "deep/a/b/ünïcödé_🔥.txt").write_text("ok")  # nested + unicode
    (n / "dup.txt").write_text("v1")
    (n / "dup.txt").write_text("v2")  # overwrite
r = s4.get(n.node_id)
print("empty:", (r / "empty.bin").read_bytes() == b"")
print("nested+unicode:", (r / "deep/a/b/ünïcödé_🔥.txt").read_text() == "ok")
print("overwrite kept last:", (r / "dup.txt").read_text() == "v2")
s4.close()

store.close()
dt_store.close()
shutil.rmtree(WORK)
print("cleaned up")

empty: True
nested+unicode: True
overwrite kept last: True
cleaned up
